# Heat Equation Solver Example

This notebook demonstrates using the `Solver`, `HeatEquation`, and `ExplicitEulerIntegrator` to evolve a Gaussian initial condition under diffusion with Dirichlet boundary conditions.

In [ ]:
# Setup and imports
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt

workspace_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pde_framework").exists():
        workspace_root = candidate
        break
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))
import pde_framework as pde

importlib.reload(pde)
print("Imported pde_framework from", workspace_root)

In [ ]:
# Build grid and initial field
from pde_framework import (
    DirichletBC,
    ExplicitEulerIntegrator,
    Grid1D,
    HeatEquation,
    ScalarField,
    Solver,
    gaussian_1d,
)

grid = Grid1D(-10.0, 10.0, 1001)
u0 = ScalarField(grid)
u0.apply_function(lambda x: gaussian_1d(x, center=0.0, sigma=0.2, amplitude=1.0))

bc = DirichletBC(0.0, 0.0)
eq = HeatEquation(alpha=0.1, bc=bc)
integrator = ExplicitEulerIntegrator()
solver = Solver(integrator, eq)

print("Initial field norm:", u0.norm())

In [ ]:
# Run solver and plot snapshots
snapshots = solver.run(u0, dt=0.0001, n_steps=200000, snapshot_stride=50000)
fig, ax = plt.subplots(figsize=(10, 6))
for idx, s in enumerate(snapshots):
    (ax.plot(grid.x, s.data, label=f"step {idx * 50000}"),)
ax.set_xlabel("x")
ax.set_xlim(-6, 6)
ax.set_ylim(0, 0.5)
ax.set_ylabel("u(x)")
ax.set_title("Heat equation diffusion (snapshots)")
ax.legend()
plt.show()
print("Final norm:", snapshots[-1].norm())